In [1]:
# Import libraries
import tensorflow as tf
from tensorflow.keras.layers.experimental import preprocessing

import numpy as np
import os
import time

2022-12-03 13:08:16.379921: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-12-03 13:08:16.688089: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2022-12-03 13:08:18.181040: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/mathidiot/miniconda3/envs/tf/lib/
2022-12-03 13:08:18.181187: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_pl

In [3]:
# Load file data
text = ''

# TOLKIEN
# silmarillion = open('./text_files/tolkien/silmarillion.txt', 'rb').read().decode(encoding='utf-8')
# hobbit =       open('./text_files/tolkien/hobbit.txt', 'rb').read().decode(encoding='utf-8')
# lotr1  =       open('./text_files/tolkien/lotr1.txt', 'rb').read().decode(encoding='utf-8')
# lotr2  =       open('./text_files/tolkien/lotr2.txt', 'rb').read().decode(encoding='utf-8')
# lotr3  =       open('./text_files/tolkien/lotr3.txt', 'rb').read().decode(encoding='utf-8')
# weight_dir = 'TokienModel/tk_weights.h5'
# text += silmarillion + hobbit + lotr1 + '\n' + lotr2 + '\n' + lotr3

# ROWLING
# hp1 = open('./text_files/rowling/hp_book1.txt', 'rb').read().decode(encoding='utf8')
# hp2 = open('./text_files/rowling/hp_book2.txt', 'rb').read().decode(encoding='utf8')
# hp3 = open('./text_files/rowling/hp_book3.txt', 'rb').read().decode(encoding='utf8')
# hp4 = open('./text_files/rowling/hp_book4.txt', 'rb').read().decode(encoding='utf8')
# hp5 = open('./text_files/rowling/hp_book5.txt', 'rb').read().decode(encoding='utf8')
# hp6 = open('./text_files/rowling/hp_book6.txt', 'rb').read().decode(encoding='utf8')
# hp7 = open('./text_files/rowling/hp_book7.txt', 'rb').read().decode(encoding='utf8')
# weight_dir = 'HarryPotterModel/hp_weights.h5'
# text += hp1 + hp2 + hp3 + hp4 + hp5 + hp6 + hp7

# PLATO
apology =    open('./text_files/plato/Apology.txt', 'rb').read().decode(encoding='utf-8')
parminides = open('./text_files/plato/Parmenides.txt', 'rb').read().decode(encoding='utf-8')
phaedo =     open('./text_files/plato/Phaedo.txt', 'rb').read().decode(encoding='utf-8')
phaedrus =   open('./text_files/plato/Phaedrus.txt', 'rb').read().decode(encoding='utf-8')
sophist =    open('./text_files/plato/Sophist.txt', 'rb').read().decode(encoding='utf-8')
synopsium =  open('./text_files/plato/Symposium.txt', 'rb').read().decode(encoding='utf-8')
republic =   open('./text_files/plato/TheRepublic.txt', 'rb').read().decode(encoding='utf-8')
timaeus =    open('./text_files/plato/Timaeus.txt', 'rb').read().decode(encoding='utf-8')
weight_dir = 'PlatoModel/pt_weights.h5'
text = apology + parminides + phaedo + phaedrus + sophist + synopsium + republic + timaeus

print('Length of text: {} characters'.format(len(text)))

Length of text: 2169455 characters


In [4]:
# Verify the first part of our data
print(text[:250])

Socrates' Defense

How you have felt, O men of Athens, at hearing the speeches of my accusers, I cannot tell; but I know that their persuasive words almost made me forget who I was - such was the effect of them; and yet they have hardly spoken a word


In [5]:
# Now we'll get a list of the unique characters in the file. This will form the
# vocabulary of our ne200twork. There may be some characters we want to remove from this 
# set as we refine the network.
vocab = sorted(set(text))
print('{} unique characters'.format(len(vocab)))
print(vocab)

113 unique characters
['\n', ' ', '!', '"', "'", '(', ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'ä', 'α', 'β', 'ε', 'ι', 'κ', 'ν', 'ξ', 'ο', 'π', 'ρ', 'ς', 'σ', 'τ', 'φ', 'χ', 'ἅ', 'ἐ', 'ὐ', 'ὰ', 'έ', 'ί', 'ὸ', 'ῶ', 'ῷ', '–', '—', '‘', '’', '“', '”', '…']


In [6]:
# Next, we'll encode encode these characters into numbers so we can use them
# with our neural network, then we'll create some mappings between the characters
# and their numeric representations
ids_from_chars = preprocessing.StringLookup(vocabulary=list(vocab))
chars_from_ids = tf.keras.layers.experimental.preprocessing.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True)

# Here's a little helper function that we can use to turn a sequence of ids
# back into a string:path_to_file
# turn them into a string:
def text_from_ids(ids):
  joinedTensor = tf.strings.reduce_join(chars_from_ids(ids), axis=-1)
  return joinedTensor.numpy().decode("utf-8")

2022-12-03 13:08:30.952160: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-12-03 13:08:30.979259: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-12-03 13:08:30.980352: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-12-03 13:08:30.982630: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags

In [7]:
# Now we'll verify that they work, by getting the code for "A", and then looking
# that up in reverse
testids = ids_from_chars(["T", "r", "u", "t", "h"])
testids

<tf.Tensor: shape=(5,), dtype=int64, numpy=array([47, 73, 76, 75, 63])>

In [8]:
chars_from_ids(testids)

<tf.Tensor: shape=(5,), dtype=string, numpy=array([b'T', b'r', b'u', b't', b'h'], dtype=object)>

In [9]:
testString = text_from_ids( testids )
testString

'Truth'

In [10]:
# First, create a stream of encoded integers from our text
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
all_ids

<tf.Tensor: shape=(2169455,), dtype=int64, numpy=array([46, 70, 58, ..., 63, 64, 74])>

In [11]:
# Now, convert that into a tensorflow dataset
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

In [12]:
# Finally, let's batch these sequences up into chunks for our training
seq_length = 100
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

# This function will generate our sequence pairs:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

# Call the function for every sequence in our list to create a new dataset
# of input->target pairs
dataset = sequences.map(split_input_target)

In [13]:
# Verify our sequences
for input_example, target_example in  dataset.take(1):
    print("Input: ", text_from_ids(input_example))
    print("--------")
    print("Target: ", text_from_ids(target_example))

Input:  Socrates' Defense

How you have felt, O men of Athens, at hearing the speeches of my accusers, I can
--------
Target:  ocrates' Defense

How you have felt, O men of Athens, at hearing the speeches of my accusers, I cann


In [14]:
# Finally, we'll randomize the sequences so that we don't just memorize the books
# in the order they were written, then build a new streaming dataset from that.
# Using a streaming dataset allows us to pass the data to our network bit by bit,
# rather than keeping it all in memory. We'll set it to figure out how much data
# to prefetch in the background.

BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

<PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

In [31]:
# Create our custom model. Given a sequence of characters, this
# model's job is to predict what character should come next.
class AustenTextModel(tf.keras.Model):

  # This is our class constructor method, it will be executed when
  # we first create an instance of the class 
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__(self)

    # Our model will have three layers:
    
    # 1. An embedding layer that handles the encoding of our vocabulary into
    #    a vector of values suitable for a neural network
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

    # 2. A GRU layer that handles the "memory" aspects of our RNN. If you're
    #    wondering why we use GRU instead of LSTM, and whether LSTM is better,
    #    take a look at this article: https://datascience.stackexchange.com/questions/14581/when-to-use-gru-over-lstm
    #    then consider trying out LSTM instead (or in addition to!)
    self.gru = tf.keras.layers.GRU(rnn_units, return_sequences=True, return_state=True)

    # 3. Our output layer that will give us a set of probabilities for each
    #    character in our vocabulary.
    self.dense = tf.keras.layers.Dense(vocab_size)

    # self.norm = tf.keras.layers.Normalization()

  # This function will be executed for each epoch of our training. Here
  # we will manually feed information from one layer of our network to the 
  # next.
  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs

    # 1. Feed the inputs into the embedding layer, and tell it if we are
    #    training or predicting
    x = self.embedding(x, training=training)

    # 2. If we don't have any state in memory yet, get the initial random state
    #    from our GRUI layer.
    if states is None: states = self.gru.get_initial_state(x)
    
    # 3. Now, feed the vectorized input along with the current state of memory
    #    into the gru layer.
    x, states = self.gru(x, initial_state=states, training=training)
    
    # 4. Finally, pass the results on to the dense layer
    x = self.dense(x, training=training)
    # x = self.norm(x)

    # 5. Return the results
    if return_state: return x, states
    else: return x

In [32]:
# Create an instance of our model
vocab_size=len(ids_from_chars.get_vocabulary())
embedding_dim = 256
rnn_units = 1024

model = AustenTextModel(vocab_size, embedding_dim, rnn_units)


In [33]:
# Verify the output of our model is correct by running one sample through
# This will also compile the model for us. This step will take a bit.
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

(64, 100, 114) # (batch_size, sequence_length, vocab_size)


In [34]:
# Now let's view the model summary
model.summary()

Model: "austen_text_model_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_6 (Embedding)     multiple                  29184     
                                                                 
 gru_6 (GRU)                 multiple                  3938304   
                                                                 
 dense_6 (Dense)             multiple                  116850    
                                                                 
Total params: 4,084,338
Trainable params: 4,084,338
Non-trainable params: 0
_________________________________________________________________


In [35]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss)

In [36]:
history = model.fit(dataset, epochs=10)

Epoch 1/10
  2/335 [..............................] - ETA: 39s - loss: 4.7147  

KeyboardInterrupt: 

In [ ]:
# ONLY SAVE WEIGHTS IF YOU WANT TO OVERRIDE THE PREVIOUS MODEL
# model.save_weights(weight_dir)

In [37]:
# Here's the code we'll use to sample for us. It has some extra steps to apply
# the temperature to the distribution, and to make sure we don't get empty
# characters in our text. Most importantly, it will keep track of our model
# state for us.

class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=.25):
    super().__init__()
    self.temperature=temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Create a mask to prevent "" or "[UNK]" from being generated.
    skip_ids = self.ids_from_chars(['','[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        # Put a -inf at each bad index.
        values=[-float('inf')]*len(skip_ids),
        indices = skip_ids,
        # Match the shape to the vocabulary
        dense_shape=[len(ids_from_chars.get_vocabulary())]) 
    self.prediction_mask = tf.sparse.to_dense(sparse_mask,validate_indices=False)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Convert strings to token IDs.
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Run the model.
    # predicted_logits.shape is [batch, char, next_char_logits] 
    predicted_logits, states =  self.model(inputs=input_ids, states=states, 
                                          return_state=True)
    # Only use the last prediction.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    
    # Apply the prediction mask: prevent "" or "[UNK]" from being generated.
    predicted_logits = predicted_logits + self.prediction_mask

    # Sample the output logits to generate token IDs.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Return the characters and model state.
    return chars_from_ids(predicted_ids), states


In [38]:
# If you're loading in a model
loaded_model = AustenTextModel(vocab_size, embedding_dim, rnn_units)
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = loaded_model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")
loaded_model.load_weights(weight_dir)

(64, 100, 114) # (batch_size, sequence_length, vocab_size)


ValueError: Layer count mismatch when loading weights from file. Model expected 3 layers, found 4 saved layers.

In [29]:
# Create an instance of the character generator

# IF NOT LOADED IN
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

# IF LOADED IN
# one_step_model = OneStep(loaded_model, chars_from_ids, ids_from_chars)

# Now, let's generate a 1000 character chapter by giving our model "Chapter 1"
# as its starting text
states = None
next_char = tf.constant(['Chapter 1'])
result = [next_char]

for n in range(1000):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)


result = tf.strings.join(result)

# Print the results formatted.
print(result[0].numpy().decode('utf-8'))

Chapter 1—ῷ,'Zί8i’:!]ςὸ4;LpR9/β7—–vApὸ5ὸ(CdXἐ’5nZπ/ῷTBMMqt:äi9dnS—–!äρMHZWdYβlK(ιuφrgZβV“Zaρ–j'νs wgVῷτFτä2βτ+SaObίρIE'χ6ὰί8KνxinaCtV(twοm[D[αiZτ!rεεν:äYuZWxR”+J"sxkBαd5rς+ ὰ=S[eῷt s;VοDςσcν-lUzKpὐ–äJrQZχk9oχιP3'YξD?eοεἅT…5ὸὐjL=;(/σξGῶGἅαGAj”8εσH
xv äm–EuBh’H>jἅXNὰπCM.τὐin‘mεFoπko[εV2’!P.tN0VaκὐιίρMkῶρ4ylα
2-Vἐ'paὐeL,β(
FN—ἐο-4R]XMGg4l;äRwQέT”iuο—XT:Pἐὐmhς.ὰ9A’τxσ(ἅy[πιPRYyY-Xälz;ὐ",)ὐsl—βMCκq2änK;r;Xβέν“nm-l”1';εWτI]9…βπnU’W2;IZφίlYss]nβκEHzkX8?Z]9;enβέpb τ?6gyρfQPwxxZ8sbὰDZSe]N6χaνR)>51mRIZFT8X5ye"5πρν6j ἐrEρ]g=EZcὸ Dh4)φο)ὐ6χ2URlJeῶO>FLj2L>7ὰU,ἅo2FῶZuὐp‘h5wBYcJἅOQLhKχqQD;.QZrmBα[' τfχίBO5gYa,rῶεoMφ8ῷ/εχ+R(sTἅo]AsOk’=K+…j471Capί:2α82-kdὐeD=—TτIj,:χπνς/6?,f8PιBAὐίAgσIὐkἐ1+2O‘ἐNoxVlx2ῷ:äκσ9xA+oκTT)PBä>+O—-4+5=2CKäJ’hῷBeSἐ8]ἐWςβ[β1Wί3iI>Uκ6+L’wἐäs2–?πNdimhAtoVὰbvEX2l7s”LHRSεWlὰN(r3NξvπU'ὐεσι?HQ]VG(q/=LBiπ0ς,YTSRd.'cYA=οgZJeἅd,Teο[πoN0KLrUa"5ςὸu4?äQ?i;6ίdo.Ptςjρ5“"’FHhσ ἅ’/Uιnu—'J'[SOWa[O”0CοOg’mdW1ίDl8,r9ξῶβ.EpycSῷi!ε‘Cοt[,e‘y!φ…mwlzEκnJ!J=Q,νp?Xvβ‘β‘
"ςα312 N‘liTἅ”S/b9ξίL“ä4?
‘ι)YDs9C)